In [0]:
# CivicPulse — Bronze Layer Ingestion
#
# Purpose:
# Ingest the raw Surat municipal complaint dataset into
# the CivicPulse Bronze Delta layer without applying
# business transformations.
#
# Bronze principle:
# Preserve the source data as closely as possible.
# Data cleaning, feature engineering, anomaly detection,
# forecasting, and intervention scoring are handled downstream.

print("CivicPulse Bronze Ingestion")
print("Spark version:", spark.version)

In [0]:
display(dbutils.fs.ls("/Workspace/CivicPulse/raw"))

In [0]:
df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Workspace/CivicPulse/raw/surat_complaints.csv")
)

In [0]:
# Preview the raw dataset
display(df_raw.limit(10))

In [0]:
# Inspect the inferred schema
df_raw.printSchema()

In [0]:
# Basic dataset dimensions
print("Rows:", df_raw.count())
print("Columns:", len(df_raw.columns))
print("Column names:", df_raw.columns)

In [0]:
# Null-value check
from pyspark.sql.functions import col, sum

display(
    df_raw.select([
        sum(col(c).isNull().cast("int")).alias(c)
        for c in df_raw.columns
    ])
)

In [0]:
# Basic range and date coverage profile
bronze_table = "workspace.civicpulse_bronze.complaints_raw"

(
    df_raw.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(bronze_table)
)

print(f"Bronze table created: {bronze_table}")

In [0]:
# Data-quality investigation:
# Test whether the reported pending count follows the assumed
# relationship:
#
# expected_pending =
# previous_pending + complaints_received - complaints_disposed
#
# This is a validation check only. The source-reported pending
# value is preserved and is not corrected using this formula.
df_bronze = spark.table("workspace.civicpulse_bronze.complaints_raw")

print("Bronze rows:", df_bronze.count())
print("Bronze columns:", df_bronze.columns)

display(df_bronze.limit(10))

In [0]:
from pyspark.sql.functions import min, max, countDistinct

profile = df_bronze.select(
    countDistinct("Date").alias("distinct_dates"),
    min("Date").alias("min_date"),
    max("Date").alias("max_date"),
    min("No_of_complain_received").alias("min_received"),
    max("No_of_complain_received").alias("max_received"),
    min("No_of_complain_disposed").alias("min_disposed"),
    max("No_of_complain_disposed").alias("max_disposed"),
    min("No_of_complain_pending").alias("min_pending"),
    max("No_of_complain_pending").alias("max_pending")
)

display(profile)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    col, lag, when, abs
)

window_spec = Window.orderBy("Date")

df_flow = (
    df_bronze
    .withColumn(
        "previous_pending",
        lag("No_of_complain_pending").over(window_spec)
    )
    .withColumn(
        "expected_pending",
        col("previous_pending")
        + col("No_of_complain_received")
        - col("No_of_complain_disposed")
    )
    .withColumn(
        "reconciliation_gap",
        col("No_of_complain_pending") - col("expected_pending")
    )
)

display(
    df_flow.select(
        "Date",
        "No_of_complain_received",
        "No_of_complain_disposed",
        "No_of_complain_pending",
        "previous_pending",
        "expected_pending",
        "reconciliation_gap"
    ).limit(20)
)

In [0]:
from pyspark.sql.functions import (
    count,
    sum,
    avg,
    max,
    min,
    abs,
    when
)

flow_summary = df_flow.select(
    count("*").alias("total_records"),
    sum(
        when(col("reconciliation_gap") != 0, 1).otherwise(0)
    ).alias("records_with_gap"),
    sum(
        when(col("reconciliation_gap") == 0, 1).otherwise(0)
    ).alias("records_without_gap"),
    avg(abs(col("reconciliation_gap"))).alias("avg_absolute_gap"),
    max(abs(col("reconciliation_gap"))).alias("max_absolute_gap")
)

display(flow_summary)

In [0]:
display(
    df_flow
    .filter(col("reconciliation_gap") == 0)
    .select(
        "Date",
        "No_of_complain_received",
        "No_of_complain_disposed",
        "No_of_complain_pending",
        "previous_pending",
        "expected_pending",
        "reconciliation_gap"
    )
)

In [0]:
display(
    df_flow
    .orderBy(abs(col("reconciliation_gap")).desc())
    .select(
        "Date",
        "No_of_complain_received",
        "No_of_complain_disposed",
        "No_of_complain_pending",
        "previous_pending",
        "expected_pending",
        "reconciliation_gap"
    )
    .limit(10)
)

In [0]:
from pyspark.sql.functions import (
    col,
    lag,
    when,
    round
)
from pyspark.sql.window import Window

window_spec = Window.orderBy("Date")

df_silver = (
    df_bronze
    .select(
        col("CityName").alias("city_name"),
        col("Date").alias("date"),
        col("No_of_complain_received").alias("complaints_received"),
        col("No_of_complain_disposed").alias("complaints_disposed"),
        col("No_of_complain_pending").alias("complaints_pending")
    )
    .withColumn(
        "previous_pending",
        lag("complaints_pending").over(window_spec)
    )
    .withColumn(
        "pending_change",
        col("complaints_pending") - col("previous_pending")
    )
    .withColumn(
        "net_complaint_flow",
        col("complaints_received") - col("complaints_disposed")
    )
    .withColumn(
        "expected_pending",
        col("previous_pending")
        + col("complaints_received")
        - col("complaints_disposed")
    )
    .withColumn(
        "reconciliation_gap",
        col("complaints_pending") - col("expected_pending")
    )
    .withColumn(
        "reconciliation_status",
        when(
            col("previous_pending").isNull(),
            "FIRST_RECORD"
        )
        .when(
            col("reconciliation_gap") == 0,
            "RECONCILED"
        )
        .otherwise(
            "RECONCILIATION_GAP"
        )
    )
)

display(df_silver.limit(20))

In [0]:
silver_table = "workspace.civicpulse_silver.complaints_daily"

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

print(f"Silver table created: {silver_table}")

In [0]:
df_silver_table = spark.table(
    "workspace.civicpulse_silver.complaints_daily"
)

print("Silver rows:", df_silver_table.count())
print("Silver columns:", len(df_silver_table.columns))

display(df_silver_table.limit(10))